write a code to import all necessary libraries, xgboost classifier, a database, drop a few columns and set the target column for classification. The data base contains of 
1. File name (string, can be dropped)
2. Class (1 = Real, 2 = Fake)
3. Features (long floating point numbers)
Then I want to define a hyperparameter list for the xgboost to go through for training and testing to find the best model based on f1 scores, accuracy, precision.
I wish to save the best model locally and then visualize the results in form of confusion matrices, graphs, and charts.

In [1]:
import pandas as pd
import numpy as np
import joblib
import xgboost as xgb
from datetime import datetime
import warnings
import json
warnings.filterwarnings('ignore')

# Sklearn imports
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve,
    precision_recall_curve, auc
)
from sklearn.preprocessing import LabelEncoder

# Visualization imports
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn-v0_8')

print("All libraries imported successfully!")

All libraries imported successfully!


In [2]:
# Configuration settings
RANDOM_STATE = 420
TEST_SIZE = 0.2

# File path - CHANGE THIS TO YOUR DATASET PATH
file_path = '../dataset/01_feature_csv/TEST_Deepfake-and-real-images-4_combined.csv'

# Quick test mode (set to True for faster execution with reduced grid)
quick_test = True  # Set to True for initial testing

print(f"Configuration:")
print(f"   Random State: {RANDOM_STATE}")
print(f"   Test Size: {TEST_SIZE}")
print(f"   Quick Test Mode: {quick_test}")


Configuration:
   Random State: 420
   Test Size: 0.2
   Quick Test Mode: True


In [3]:
print("Loading dataset...")

# Load data (handles multiple formats)
try:
    df = pd.read_csv(file_path)
    print("   Loaded as CSV")
except:
    try:
        df = pd.read_excel(file_path)
        print("   Loaded as Excel")
    except:
        df = pd.read_json(file_path)
        print("   Loaded as JSON")

print(f"   Dataset shape: {df.shape}")
print(f"   Columns: {list(df.columns)}")

# Display first few rows
print("\nFirst 5 rows:")
display(df.head())

# Basic dataset info
print(f"\nDataset Information:")
print(f"   Total rows: {len(df):,}")
print(f"   Total columns: {len(df.columns)}")
print(f"   Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Loading dataset...
   Loaded as CSV
   Dataset shape: (35304, 2051)
   Columns: ['filename', 'class', 'image_path', 'feature_0', 'feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7', 'feature_8', 'feature_9', 'feature_10', 'feature_11', 'feature_12', 'feature_13', 'feature_14', 'feature_15', 'feature_16', 'feature_17', 'feature_18', 'feature_19', 'feature_20', 'feature_21', 'feature_22', 'feature_23', 'feature_24', 'feature_25', 'feature_26', 'feature_27', 'feature_28', 'feature_29', 'feature_30', 'feature_31', 'feature_32', 'feature_33', 'feature_34', 'feature_35', 'feature_36', 'feature_37', 'feature_38', 'feature_39', 'feature_40', 'feature_41', 'feature_42', 'feature_43', 'feature_44', 'feature_45', 'feature_46', 'feature_47', 'feature_48', 'feature_49', 'feature_50', 'feature_51', 'feature_52', 'feature_53', 'feature_54', 'feature_55', 'feature_56', 'feature_57', 'feature_58', 'feature_59', 'feature_60', 'feature_61', 'feature_62', 'feature_63'

,filename,class,image_path,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,...,feature_2038,feature_2039,feature_2040,feature_2041,feature_2042,feature_2043,feature_2044,feature_2045,feature_2046,feature_2047
0,image_0.png,0,image_0.png,0.066736,0.398247,0.035540,0.697569,0.333106,0.581918,0.077922,...,0.720562,0.058847,0.050144,1.308185,0.421481,0.802583,0.026240,1.170586,0.344675,0.780771
1,image_1.png,0,image_1.png,0.518612,0.542352,0.445812,0.203894,0.267886,0.508992,0.150006,...,0.693732,0.027530,0.101434,0.931467,0.053287,0.676475,0.034142,0.189280,0.137137,0.449806
2,image_10.png,0,image_10.png,0.059122,0.391779,0.138042,0.550379,0.199735,0.416898,0.138010,...,0.483257,0.130117,0.042231,0.938167,0.023627,1.131172,0.077605,0.454639,0.061200,0.657048
3,image_100.png,0,image_100.png,0.133516,0.516427,0.042039,0.393566,0.916419,0.454148,0.285654,...,0.690564,0.021676,0.146509,0.565516,0.069728,1.127737,0.296589,0.540830,0.136188,0.980616
4,image_1000.png,0,image_1000.png,0.219320,0.443218,0.067321,0.627571,0.598842,0.251692,0.605568,...,1.248132,0.001825,0.078800,1.939213,0.916627,0.611084,0.083661,1.133962,0.090683,0.251972



Dataset Information:
   Total rows: 35,304
   Total columns: 2051
   Memory usage: 556.72 MB


In [4]:
# 3. Preprocess: Remove first column (file path), separate features and labels
columns_to_drop = ['filename', 'class', 'image_path']
X = df.drop(columns=columns_to_drop, axis=1, inplace=False)  # Features
y = df['class']      # Target column

print(f"   Features shape: {X.shape}")
print(f"   Target shape: {y.shape}")

# Convert target to binary classification
original_classes = y.unique()
print(f"   Original classes: {original_classes}")

   Features shape: (35304, 2048)
   Target shape: (35304,)
   Original classes: [0 1]


In [5]:
# Check for missing values
missing_features = X.isnull().sum().sum()
missing_target = y.isnull().sum()

print(f"\nMissing Values Check:")
print(f"   Missing values in features: {missing_features}")
print(f"   Missing values in target: {missing_target}")

if missing_features > 0:
    print("     Consider handling missing values before proceeding")

# Display class distribution
print(f"\nClass Distribution:")
class_counts = y.value_counts().sort_index()
for class_val, count in class_counts.items():
    class_name = "Real" if class_val == 0 else "Fake"
    percentage = count/len(y)*100
    print(f"   {class_name} ({class_val}): {count:,} samples ({percentage:.1f}%)")
    
# Check for class imbalance
class_ratio = class_counts.max() / class_counts.min()
print(f"   Class imbalance ratio: {class_ratio:.2f}:1")
if class_ratio > 2:
    print("     Dataset is imbalanced - will use scale_pos_weight in XGBoost")


Missing Values Check:
   Missing values in features: 0
   Missing values in target: 0

Class Distribution:
   Real (0): 17,640 samples (50.0%)
   Fake (1): 17,664 samples (50.0%)
   Class imbalance ratio: 1.00:1


In [6]:
print("Splitting data into train and test sets...")

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=TEST_SIZE, 
    random_state=RANDOM_STATE, 
    stratify=y
)

print(f"   Training set: {X_train.shape} samples")
print(f"   Test set: {X_test.shape} samples")

# Verify class distribution is preserved
print(f"\nClass distribution after split:")
print("Training set:")
train_counts = y_train.value_counts().sort_index()
for class_val, count in train_counts.items():
    class_name = "Real" if class_val == 0 else "Fake"
    percentage = count/len(y_train)*100
    print(f"   {class_name}: {count:,} ({percentage:.1f}%)")

print("Test set:")
test_counts = y_test.value_counts().sort_index()
for class_val, count in test_counts.items():
    class_name = "Real" if class_val == 0 else "Fake"
    percentage = count/len(y_test)*100
    print(f"   {class_name}: {count:,} ({percentage:.1f}%)")

Splitting data into train and test sets...
   Training set: (28243, 2048) samples
   Test set: (7061, 2048) samples

Class distribution after split:
Training set:
   Real: 14,112 (50.0%)
   Fake: 14,131 (50.0%)
Test set:
   Real: 3,528 (50.0%)
   Fake: 3,533 (50.0%)


In [7]:
print("Defining hyperparameter grid...")

if quick_test:
    # Reduced grid for quick testing
    param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [3, 6],
        'learning_rate': [0.1, 0.2],
        'subsample': [0.8, 1.0],
        'min_child_weight': [1, 3],
        'scale_pos_weight': [1, 2]
    }
    print("Using reduced parameter grid for quick testing")
else:
    # Full comprehensive grid
    param_grid = {
        # Core parameters
        'n_estimators': [100, 200, 300, 500],           # Number of trees
        'max_depth': [3, 4, 6, 8],                      # Tree depth
        'learning_rate': [0.01, 0.05, 0.1, 0.2],       # Step size
        
        # Regularization
        'min_child_weight': [1, 3, 5],                  # Minimum child weight
        'gamma': [0, 0.1, 0.2],                        # Minimum loss reduction
        'reg_alpha': [0, 0.1, 0.5],                    # L1 regularization
        'reg_lambda': [1, 1.5, 2],                     # L2 regularization
        
        # Sampling
        'subsample': [0.8, 0.9, 1.0],                  # Row sampling
        'colsample_bytree': [0.8, 0.9, 1.0],          # Feature sampling
        
        # For imbalanced classes
        'scale_pos_weight': [1, 2, 3]                  # Handle class imbalance
    }
    print("Using comprehensive parameter grid")

# Calculate total combinations
total_combinations = 1
for param, values in param_grid.items():
    total_combinations *= len(values)

print(f"   Total parameter combinations: {total_combinations:,}")
print("   Parameters to tune:")
for param, values in param_grid.items():
    print(f"     {param}: {values}")

if total_combinations > 1000:
    print(f"     This will take considerable time! Consider using quick_test=True")

Defining hyperparameter grid...
Using reduced parameter grid for quick testing
   Total parameter combinations: 64
   Parameters to tune:
     n_estimators: [100, 200]
     max_depth: [3, 6]
     learning_rate: [0.1, 0.2]
     subsample: [0.8, 1.0]
     min_child_weight: [1, 3]
     scale_pos_weight: [1, 2]


In [10]:
print("Initializing XGBoost model and GridSearchCV...")

# Initialize XGBoost classifier
xgb_model = xgb.XGBClassifier(
    tree_method='hist',      # CPU-optimized
    device='cpu',            # CPU only
    n_jobs=-2,              # Use all cores
    random_state=RANDOM_STATE,
    eval_metric='logloss'    # Evaluation metric
)

# Set up scoring metrics
scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}

# Initialize GridSearchCV
grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    scoring=scoring,
    cv=2,
    refit='f1',              # Optimize for F1-score
    n_jobs=-1,              # Use all cores
    verbose=1               # Show progress
)

print("GridSearchCV initialized")
print(f"   Optimization metric: F1-score")
print(f"   Scoring metrics: {list(scoring.keys())}")

Initializing XGBoost model and GridSearchCV...
GridSearchCV initialized
   Optimization metric: F1-score
   Scoring metrics: ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']


In [11]:
print("Starting XGBoost training with GridSearchCV...")

# Record start time
start_time = datetime.now()
print(f"   Started at: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")

# Fit the model
grid_search.fit(X_train, y_train)

# Record end time
end_time = datetime.now()
training_time = end_time - start_time

print(f"   Training completed at: {end_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"   Total training time: {training_time}")

# Get best model
best_model = grid_search.best_estimator_

print(f"\nBest Parameters Found:")
for param, value in grid_search.best_params_.items():
    print(f"   {param}: {value}")

print(f"\nBest Cross-Validation Scores:")
best_index = grid_search.best_index_
for metric in scoring.keys():
    score = grid_search.cv_results_[f'mean_test_{metric}'][best_index]
    std = grid_search.cv_results_[f'std_test_{metric}'][best_index]
    print(f"   {metric.upper()}: {score:.4f} (±{std*2:.4f})")

print(f"\nBest model selected and ready for evaluation")

Starting XGBoost training with GridSearchCV...
   Started at: 2025-09-01 16:17:55
Fitting 2 folds for each of 64 candidates, totalling 128 fits


KeyboardInterrupt: 

In [ ]:
print("Evaluating model performance...")

# Make predictions
y_train_pred = best_model.predict(X_train)
y_test_pred = best_model.predict(X_test)
y_test_proba = best_model.predict_proba(X_test)[:, 1]

# Calculate metrics
metrics = {}

# Training metrics
metrics['train_accuracy'] = accuracy_score(y_train, y_train_pred)
metrics['train_precision'] = precision_score(y_train, y_train_pred)
metrics['train_recall'] = recall_score(y_train, y_train_pred)
metrics['train_f1'] = f1_score(y_train, y_train_pred)

# Test metrics
metrics['test_accuracy'] = accuracy_score(y_test, y_test_pred)
metrics['test_precision'] = precision_score(y_test, y_test_pred)
metrics['test_recall'] = recall_score(y_test, y_test_pred)
metrics['test_f1'] = f1_score(y_test, y_test_pred)
metrics['test_roc_auc'] = roc_auc_score(y_test, y_test_proba)

# Print results
print("\nPERFORMANCE METRICS:")
print("=" * 50)
print(f"{'Metric':<15} {'Training':<12} {'Test':<12}")
print("=" * 50)
print(f"{'Accuracy':<15} {metrics['train_accuracy']:<12.4f} {metrics['test_accuracy']:<12.4f}")
print(f"{'Precision':<15} {metrics['train_precision']:<12.4f} {metrics['test_precision']:<12.4f}")
print(f"{'Recall':<15} {metrics['train_recall']:<12.4f} {metrics['test_recall']:<12.4f}")
print(f"{'F1-Score':<15} {metrics['train_f1']:<12.4f} {metrics['test_f1']:<12.4f}")
print(f"{'ROC-AUC':<15} {'':<12} {metrics['test_roc_auc']:<12.4f}")
print("=" * 50)

# Classification report
print("\nDETAILED CLASSIFICATION REPORT:")
print(classification_report(y_test, y_test_pred, target_names=['Real', 'Fake']))

# Check for overfitting
accuracy_diff = metrics['train_accuracy'] - metrics['test_accuracy']
f1_diff = metrics['train_f1'] - metrics['test_f1']

print(f"\nOverfitting Check:")
print(f"   Accuracy difference (train-test): {accuracy_diff:.4f}")
print(f"   F1-score difference (train-test): {f1_diff:.4f}")

if accuracy_diff > 0.1 or f1_diff > 0.1:
    print("   Possible overfitting detected (>0.1 difference)")
elif accuracy_diff > 0.05 or f1_diff > 0.05:
    print("   Mild overfitting (>0.05 difference)")
else:
    print("   Good generalization (minimal overfitting)")

In [ ]:
print("Saving model and metrics...")

# Create timestamp for file naming
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Save model
model_filename = f"05_xgboost_model_{timestamp}.joblib"
joblib.dump(best_model, model_filename)
print(f"   Model saved: {model_filename}")

# Save metrics
metrics_filename = f"05_xgboost_model_metrics_{timestamp}.json"
with open(metrics_filename, 'w') as f:
    json.dump(metrics, f, indent=4)
print(f"   Metrics saved: {metrics_filename}")

# Save best parameters
params_filename = f"05_xgboost_model_params_{timestamp}.json"
with open(params_filename, 'w') as f:
    json.dump(grid_search.best_params_, f, indent=4)
print(f"   Best parameters saved: {params_filename}")

# Save grid search results summary
results_summary = {
    'best_score': grid_search.best_score_,
    'best_params': grid_search.best_params_,
    'cv_results_summary': {
        'mean_test_accuracy': grid_search.cv_results_['mean_test_accuracy'][best_index],
        'mean_test_precision': grid_search.cv_results_['mean_test_precision'][best_index],
        'mean_test_recall': grid_search.cv_results_['mean_test_recall'][best_index],
        'mean_test_f1': grid_search.cv_results_['mean_test_f1'][best_index],
        'mean_test_roc_auc': grid_search.cv_results_['mean_test_roc_auc'][best_index]
    },
    'training_time': str(training_time),
    'timestamp': timestamp
}

summary_filename = f"05_xgboost_grid_search_summary_{timestamp}.json"
with open(summary_filename, 'w') as f:
    json.dump(results_summary, f, indent=4)
print(f"   Grid search summary saved: {summary_filename}")

print(f"\nAll files saved with timestamp: {timestamp}")

In [ ]:
print("Creating comprehensive visualizations...")

# Set up the plotting area
fig = plt.figure(figsize=(20, 15))

# 1. Confusion Matrix
plt.subplot(2, 4, 1)
cm = confusion_matrix(y_test, y_test_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')

# 2. ROC Curve
plt.subplot(2, 4, 2)
fpr, tpr, _ = roc_curve(y_test, y_test_proba)
roc_auc = auc(fpr, tpr)
plt.plot(fpr, tpr, color='darkorange', lw=2, 
         label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve', fontsize=14, fontweight='bold')
plt.legend(loc="lower right")

# 3. Precision-Recall Curve
plt.subplot(2, 4, 3)
precision, recall, _ = precision_recall_curve(y_test, y_test_proba)
pr_auc = auc(recall, precision)
plt.plot(recall, precision, color='red', lw=2,
         label=f'PR curve (AUC = {pr_auc:.3f})')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve', fontsize=14, fontweight='bold')
plt.legend(loc="lower left")

# 4. Feature Importance (Top 15)
plt.subplot(2, 4, 4)
importance = best_model.feature_importances_
feature_names = [f'Feature_{i}' for i in range(len(importance))]
indices = np.argsort(importance)[::-1][:15]  # Top 15 features

plt.bar(range(15), importance[indices])
plt.title('Top 15 Feature Importance', fontsize=14, fontweight='bold')
plt.xlabel('Features')
plt.ylabel('Importance Score')
plt.xticks(range(15), [feature_names[i] for i in indices], rotation=45, ha='right')

# 5. Class Distribution
plt.subplot(2, 4, 5)
class_counts = pd.Series(y_train).value_counts().sort_index()
labels = ['Real', 'Fake']
colors = ['lightblue', 'lightcoral']
plt.pie(class_counts.values, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
plt.title('Training Data\nClass Distribution', fontsize=14, fontweight='bold')

# 6. Prediction Confidence Distribution
plt.subplot(2, 4, 6)
plt.hist(y_test_proba, bins=30, alpha=0.7, color='skyblue', edgecolor='black')
plt.xlabel('Prediction Confidence (Fake Class)')
plt.ylabel('Frequency')
plt.title('Prediction Confidence\nDistribution', fontsize=14, fontweight='bold')
plt.axvline(x=0.5, color='red', linestyle='--', alpha=0.7, label='Decision Threshold')
plt.legend()

# 7. Cumulative Feature Importance
plt.subplot(2, 4, 7)
importance_sorted = np.sort(importance)[::-1]
cumsum_importance = np.cumsum(importance_sorted)
plt.plot(range(1, len(cumsum_importance) + 1), cumsum_importance, 'b-', linewidth=2)
plt.xlabel('Number of Features')
plt.ylabel('Cumulative Importance')
plt.title('Cumulative Feature\nImportance', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)

# Add horizontal lines for reference
plt.axhline(y=0.8, color='red', linestyle='--', alpha=0.5, label='80%')
plt.axhline(y=0.9, color='orange', linestyle='--', alpha=0.5, label='90%')
plt.legend()

# 8. Model Performance Summary
plt.subplot(2, 4, 8)
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
test_scores = [
    metrics['test_accuracy'],
    metrics['test_precision'],
    metrics['test_recall'],
    metrics['test_f1']
]

bars = plt.bar(metrics_names, test_scores, color=['skyblue', 'lightgreen', 'gold', 'lightcoral'])
plt.ylim(0, 1)
plt.title('Model Performance\nSummary', fontsize=14, fontweight='bold')
plt.ylabel('Score')

# Add value labels on bars
for bar, score in zip(bars, test_scores):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{score:.3f}', ha='center', va='bottom', fontweight='bold')

plt.xticks(rotation=45)

# Adjust layout and save
plt.tight_layout()
viz_filename = f"xgboost_analysis_{timestamp}.png"
plt.savefig(viz_filename, dpi=300, bbox_inches='tight')
print(f"   ✅ Visualizations saved: {viz_filename}")

# Display the plot
plt.show()

In [ ]:
print("Detailed Feature Importance Analysis...")

# Create feature importance DataFrame
feature_importance_df = pd.DataFrame({
    'feature': [f'Feature_{i}' for i in range(len(best_model.feature_importances_))],
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\nTop 20 Most Important Features:")
print("=" * 40)
for i, (_, row) in enumerate(feature_importance_df.head(20).iterrows(), 1):
    print(f"{i:2d}. {row['feature']:<15}: {row['importance']:.6f}")

# Calculate cumulative importance
feature_importance_df['cumulative_importance'] = feature_importance_df['importance'].cumsum()

# Find features that contribute to 80% and 90% of total importance
features_80 = len(feature_importance_df[feature_importance_df['cumulative_importance'] <= 0.8])
features_90 = len(feature_importance_df[feature_importance_df['cumulative_importance'] <= 0.9])

print(f"\nFeature Importance Summary:")
print(f"   Features contributing to 80% importance: {features_80} ({features_80/len(feature_importance_df)*100:.1f}%)")
print(f"   Features contributing to 90% importance: {features_90} ({features_90/len(feature_importance_df)*100:.1f}%)")

# Save feature importance to CSV
importance_filename = f"feature_importance_{timestamp}.csv"
feature_importance_df.to_csv(importance_filename, index=False)
print(f"   Feature importance saved: {importance_filename}")

# Create individual feature importance plot
plt.figure(figsize=(12, 8))
top_features = feature_importance_df.head(20)
plt.barh(range(len(top_features)), top_features['importance'])
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Importance Score')
plt.title('Top 20 Feature Importance', fontsize=16, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)

# Add value labels
for i, v in enumerate(top_features['importance']):
    plt.text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=9)

plt.tight_layout()
feature_plot_filename = f"feature_importance_detailed_{timestamp}.png"
plt.savefig(feature_plot_filename, dpi=300, bbox_inches='tight')
print(f"   Detailed feature importance plot saved: {feature_plot_filename}")
plt.show()

In [ ]:
print("PIPELINE COMPLETED SUCCESSFULLY!")
print("=" * 60)

print(f"Files Created:")
print(f"   • Model: {model_filename}")
print(f"   • Metrics: {metrics_filename}")
print(f"   • Parameters: {params_filename}")
print(f"   • Grid Search Summary: {summary_filename}")
print(f"   • Feature Importance: {importance_filename}")
print(f"   • Visualizations: {viz_filename}")
print(f"   • Feature Importance Plot: {feature_plot_filename}")

print(f"\nFinal Model Performance:")
print(f"   • F1-Score: {metrics['test_f1']:.4f}")
print(f"   • Accuracy: {metrics['test_accuracy']:.4f}")
print(f"   • Precision: {metrics['test_precision']:.4f}")
print(f"   • Recall: {metrics['test_recall']:.4f}")
print(f"   • ROC-AUC: {metrics['test_roc_auc']:.4f}")

print(f"\nTraining Summary:")
print(f"   • Total training time: {training_time}")
print(f"   • Parameter combinations tested: {total_combinations:,}")

print(f"\nBest Hyperparameters:")
for param, value in grid_search.best_params_.items():
    print(f"   • {param}: {value}")

print(f"\nDataset Summary:")
print(f"   • Total samples: {len(X):,}")
print(f"   • Features: {X.shape:,}")
print(f"   • Training samples: {len(X_train):,}")
print(f"   • Test samples: {len(X_test):,}")

print(f"\nTo use the saved model:")
print(f"   loaded_model = joblib.load('{model_filename}')")
print(f"   predictions = loaded_model.predict(new_data)")
print(f"   probabilities = loaded_model.predict_proba(new_data)")